# 多维评分预测（Multi-Dimensional Rating）

对应 `docs/multid.md`：结构化特征 + 评论弱监督 aspect 目标，基线 → Ridge → 随机森林。

**验证习惯**：每个 Step 后运行「断言 / 打印检查」，确认无误再往下。

## Step 0 — 路径与依赖

- 默认使用仓库内 `data/slice_representative/`（小样本、可快速跑通）。
- 换全量数据时改 `DATA_ROOT` 为 `data/cleaned/` 即可。

In [50]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:

    def display(x):
        print(x)


# 使 notebooks/multi 下的 multid_prep 可导入
ROOT = Path.cwd()
if (ROOT / "multid_prep.py").exists():
    sys.path.insert(0, str(ROOT))
elif (ROOT / "notebooks" / "multi" / "multid_prep.py").exists():
    sys.path.insert(0, str(ROOT / "notebooks" / "multi"))
elif (ROOT.parent / "notebooks" / "multi" / "multid_prep.py").exists():
    sys.path.insert(0, str(ROOT.parent / "notebooks" / "multi"))

import importlib

import multid_prep

importlib.reload(multid_prep)
import multid_prep as mp

print("multid_prep 路径:", getattr(mp, "__file__", "?"))
assert hasattr(mp, "build_review_signal_features"), (
    "multid_prep 版本过旧或路径错误，请确认 __file__ 指向仓库内 notebooks/multi/multid_prep.py，"
    "并重新运行本格。"
)


def _find_data_root(start: Path) -> Path:
    for base in [start, *start.parents]:
        cand = base / "data" / "slice_representative" / "business_dining.csv"
        if cand.is_file():
            return base / "data" / "slice_representative"
    return start / "data" / "slice_representative"


DATA_ROOT = _find_data_root(ROOT)

BUSINESS_CSV = DATA_ROOT / "business_dining.csv"
REVIEW_CSV = DATA_ROOT / "review_dining.csv"

print("DATA_ROOT =", DATA_ROOT.resolve())
assert BUSINESS_CSV.exists(), f"缺少 {BUSINESS_CSV}"
assert REVIEW_CSV.exists(), f"缺少 {REVIEW_CSV}"

multid_prep 路径: /Users/guyu/Desktop/ml final project/commercial-dining-intelligence/notebooks/multi/multid_prep.py
DATA_ROOT = /Users/guyu/Desktop/ml final project/commercial-dining-intelligence/data/slice_representative


## Step 1 — 加载原始 CSV（验证行数与主键）

In [51]:
biz_raw = mp.load_csv_business(BUSINESS_CSV)
rev_raw = mp.load_csv_reviews(REVIEW_CSV)

print("business 形状:", biz_raw.shape)
print("review 形状:", rev_raw.shape)
print("business 列:", list(biz_raw.columns))
assert biz_raw["business_id"].duplicated().sum() == 0, "business_id 应唯一"
assert rev_raw["review_id"].duplicated().sum() == 0, "review_id 应唯一"
biz_raw[["business_id", "stars", "city", "state"]].head(3)

business 形状: (2500, 14)
review 形状: (14965, 9)
business 列: ['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attributes', 'categories', 'hours']


,business_id,stars,city,state
0,H5cH1BbNfP5YEkytAQaYrg,1.5,Edmonton,AB
1,WzWBfCHurRUGpQuXkMnfow,2.0,Edmonton,AB
2,pPe3lGGDgrghGbSPqUtKJw,2.0,Edmonton,AB


## Step 2 — 清理与列展开（attributes / categories / hours）

**验证点**：展开后出现 `attr_*`、`categories_count`、`hours_days_open` 等；缺失率合理。

In [52]:
biz = mp.clean_and_expand_business(biz_raw)
print("展开后形状:", biz.shape)
expand_cols = [c for c in biz.columns if c.startswith("attr_") or c in ("categories_count", "categories_text", "category_primary", "hours_days_open")]
print("展开列示例:", expand_cols)
display(biz[expand_cols].head(3))
print(biz[["stars", "attr_price_range", "categories_count", "hours_days_open"]].describe())

assert "attr_price_range" in biz.columns
assert biz["business_id"].isna().sum() == 0

展开后形状: (2500, 27)
展开列示例: ['attr_price_range', 'attr_good_for_kids', 'attr_takeout', 'attr_delivery', 'attr_outdoor_seating', 'attr_credit_card', 'attr_reservations', 'attr_good_for_groups', 'attr_wifi_indicator', 'categories_count', 'category_primary', 'categories_text', 'hours_days_open']


,attr_price_range,attr_good_for_kids,attr_takeout,attr_delivery,attr_outdoor_seating,attr_credit_card,attr_reservations,attr_good_for_groups,attr_wifi_indicator,categories_count,category_primary,categories_text,hours_days_open
0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,5,Restaurants,restaurants food coffee & tea fast food burgers,0.0
1,3.0,0.0,NaN,0.0,0.0,NaN,NaN,1.0,0.0,3,Latin American,latin american restaurants mexican,NaN
2,2.0,1.0,1.0,1.0,NaN,NaN,0.0,1.0,0.0,2,Restaurants,restaurants fast food,NaN


             stars  attr_price_range  categories_count  hours_days_open
count  2500.000000       2123.000000       2500.000000      2162.000000
mean      3.555400          1.612341          4.643200         6.117484
std       0.842683          0.597654          2.335568         1.503991
min       1.000000          1.000000          1.000000         0.000000
25%       3.000000          1.000000          3.000000         6.000000
50%       3.500000          2.000000          4.000000         7.000000
75%       4.000000          2.000000          6.000000         7.000000
max       5.000000          4.000000         16.000000         7.000000


## Step 3 — 弱监督 aspect 目标（关键词 + 评论星级 + 情感）

**验证点**（见 `docs/multid.md` §4、§验证）：
- 每个商户一行目标；无评论的商户回退为 overall `stars`。
- `corr(target_*, stars)` 应为正（高整体分 → aspect 代理分偏高）。

In [53]:
targets = mp.build_weak_aspect_targets(rev_raw, biz[["business_id", "stars"]])
print("目标表形状:", targets.shape)
print(mp.verify_merge(biz, targets))
corrs = mp.verify_weak_targets(biz, targets)
print("与 overall 的 Pearson 相关:", corrs)
for k, v in corrs.items():
    assert v > 0.3, f"{k} 过低，请检查关键词或数据"

hit_cols = [c for c in targets.columns if c.endswith("_keyword_hits")]
print("关键词命中条数（各 aspect）描述统计:")
display(targets[hit_cols].describe())

目标表形状: (2500, 9)
{'n_business': 2500, 'n_targets': 2500, 'targets_without_business': 0, 'business_without_targets': 0}
与 overall 的 Pearson 相关: {'corr_food_vs_overall': 0.9878267452742813, 'corr_service_vs_overall': 0.9828927774195353, 'corr_atmosphere_vs_overall': 0.9816263303336716, 'corr_value_vs_overall': 0.976676533202059}
关键词命中条数（各 aspect）描述统计:


,food_keyword_hits,service_keyword_hits,atmosphere_keyword_hits,value_keyword_hits
count,2500.000000,2500.00000,2500.000000,2500.000000
mean,4.814400,3.22200,1.126400,1.697200
std,37.224753,25.61337,12.386371,13.776176
min,0.000000,0.00000,0.000000,0.000000
25%,0.000000,0.00000,0.000000,0.000000
50%,0.000000,0.00000,0.000000,0.000000
75%,0.000000,0.00000,0.000000,0.000000
max,915.000000,624.00000,431.000000,343.000000


## Step 4 — 合并建模表 & 定义特征 / 目标列

In [54]:
model_df = biz.merge(targets, on="business_id", how="inner", validate="one_to_one")

rev_sig = mp.build_review_signal_features(rev_raw, biz[["business_id", "stars"]])
model_df = model_df.merge(rev_sig, on="business_id", how="left", validate="one_to_one")
REV_SIGNAL_COLS = [
    "rev_neg_word_density",
    "rev_complaint_hit_rate",
    "rev_low_review_star_share",
    "rev_high_review_star_share",
    "rev_mean_sentiment",
    "rev_harsh_word_density",
    "rev_harsh_review_share",
    "rev_one_star_share",
    "rev_max_neg_density",
    "rev_neg_density_std",
]
for c in REV_SIGNAL_COLS:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce").fillna(0.0)

# 低评论量但负面相对集中（仅用该行 review_count，无全局统计泄漏）
rc = model_df["review_count"].clip(lower=0).astype(float)
model_df["feat_neg_concentration"] = model_df["rev_neg_word_density"] / (np.log1p(rc) + 1.0)

print("合并后:", model_df.shape)
print("评论负面/投诉信号（按店）:")
display(model_df[REV_SIGNAL_COLS].describe())

TARGETS = ["stars", "target_food", "target_service", "target_atmosphere", "target_value"]
NUMERIC_FEATURES = [
    "latitude",
    "longitude",
    "review_count",
    "is_open",
    "attr_price_range",
    "attr_good_for_kids",
    "attr_takeout",
    "attr_delivery",
    "attr_outdoor_seating",
    "attr_credit_card",
    "attr_reservations",
    "attr_good_for_groups",
    "attr_wifi_indicator",
    "categories_count",
    "hours_days_open",
] + REV_SIGNAL_COLS + ["feat_neg_concentration"]

for c in NUMERIC_FEATURES:
    assert c in model_df.columns, f"缺少特征列 {c}"

model_df[TARGETS].head()

合并后: (2500, 46)
评论负面/投诉信号（按店）:


,rev_neg_word_density,rev_complaint_hit_rate,rev_low_review_star_share,rev_high_review_star_share,rev_mean_sentiment,rev_harsh_word_density,rev_harsh_review_share,rev_one_star_share,rev_max_neg_density,rev_neg_density_std
count,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000,2500.000000
mean,0.000447,0.009092,0.022383,0.059136,0.039205,0.000428,0.019958,0.014904,0.004292,0.000932
std,0.002135,0.044711,0.096678,0.200209,0.139032,0.002091,0.078473,0.071354,0.019671,0.004032
min,0.000000,0.000000,0.000000,0.000000,-0.342424,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,0.033102,0.666667,1.000000,1.000000,0.772222,0.033102,0.818182,1.000000,0.300000,0.065975


,stars,target_food,target_service,target_atmosphere,target_value
0,1.5,1.500,1.500,1.5,1.500
1,2.0,2.925,4.125,2.0,2.775
2,2.0,2.000,2.000,2.0,2.000
3,2.0,2.000,2.000,2.0,2.000
4,2.0,2.000,2.000,2.0,2.000


## Step 5 — 按 `business_id` 划分 train / val / test（无泄漏）

**验证点**：三组 `business_id` 交集为空。

In [55]:
rng = np.random.RandomState(42)
ids = model_df["business_id"].unique()
rng.shuffle(ids)
n = len(ids)
n_train = int(0.6 * n)
n_val = int(0.2 * n)
train_ids = set(ids[:n_train])
val_ids = set(ids[n_train : n_train + n_val])
test_ids = set(ids[n_train + n_val :])

assert train_ids.isdisjoint(val_ids) and train_ids.isdisjoint(test_ids) and val_ids.isdisjoint(test_ids)

train_df = model_df[model_df["business_id"].isin(train_ids)].copy()
val_df = model_df[model_df["business_id"].isin(val_ids)].copy()
test_df = model_df[model_df["business_id"].isin(test_ids)].copy()
print("train", len(train_df), "val", len(val_df), "test", len(test_df))

train 1500 val 500 test 500


## Step 6 — 构造特征矩阵（数值填充 + 州 one-hot + 类别 TF-IDF + SVD 稠密化）

**验证点**：训练矩阵无 NaN；形状与样本数一致。

说明：随机森林不接受稀疏输入，故对 `categories_text` 使用 `TruncatedSVD` 得到稠密向量。

In [56]:
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def build_Xy(df: pd.DataFrame):
    X_num = df[NUMERIC_FEATURES].copy()
    imp = SimpleImputer(strategy="median")
    X_num_imp = pd.DataFrame(imp.fit_transform(X_num), columns=NUMERIC_FEATURES, index=df.index)
    scaler = StandardScaler()
    X_num_s = scaler.fit_transform(X_num_imp)

    state_dummies = pd.get_dummies(df["state"].fillna("UNK"), prefix="state")
    text = df["categories_text"].fillna("")
    tfidf = TfidfVectorizer(max_features=120, min_df=2, ngram_range=(1, 2))
    Xt = tfidf.fit_transform(text)
    svd = TruncatedSVD(n_components=24, random_state=42)
    X_txt = svd.fit_transform(Xt)

    X = np.hstack([X_num_s, state_dummies.to_numpy(dtype=float), X_txt])
    y = df[TARGETS].to_numpy(dtype=float)
    return X, y, {
        "imputer": imp,
        "scaler": scaler,
        "tfidf": tfidf,
        "svd": svd,
        "state_columns": state_dummies.columns.tolist(),
    }


X_train, y_train, art_train = build_Xy(train_df)


def transform_X(df: pd.DataFrame, art):
    X_num = df[NUMERIC_FEATURES].copy()
    X_num_imp = pd.DataFrame(art["imputer"].transform(X_num), columns=NUMERIC_FEATURES, index=df.index)
    X_num_s = art["scaler"].transform(X_num_imp)
    state = df["state"].fillna("UNK")
    st_cols = art["state_columns"]
    dummies = pd.get_dummies(state, prefix="state")
    for c in st_cols:
        if c not in dummies.columns:
            dummies[c] = 0.0
    dummies = dummies[st_cols]
    text = df["categories_text"].fillna("")
    Xt = art["tfidf"].transform(text)
    X_txt = art["svd"].transform(Xt)
    return np.hstack([X_num_s, dummies.to_numpy(dtype=float), X_txt])


X_val = transform_X(val_df, art_train)
X_test = transform_X(test_df, art_train)
y_val = val_df[TARGETS].to_numpy(dtype=float)
y_test = test_df[TARGETS].to_numpy(dtype=float)

assert np.isfinite(X_train).all() and np.isfinite(X_val).all() and np.isfinite(X_test).all()
print("X_train", X_train.shape, "y_train", y_train.shape)

X_train (1500, 64) y_train (1500, 5)


## Step 7 — 基线（训练集目标均值）与 Ridge 多输出

**验证点**：Ridge 的 test MAE 应 **≤** 常数基线（至少 overall）。

In [57]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.multioutput import MultiOutputRegressor

train_mean = y_train.mean(axis=0)
baseline_pred = np.tile(train_mean, (len(y_test), 1))


def report_mae(y_true, y_pred, title: str):
    out = {}
    for j, name in enumerate(TARGETS):
        out[name] = mean_absolute_error(y_true[:, j], y_pred[:, j])
    out["macro"] = float(np.mean(list(out.values())))
    print(title, out)
    return out


mae_test_baseline = report_mae(y_test, baseline_pred, "基线 MAE (train mean)")

ridge = MultiOutputRegressor(Ridge(alpha=2.0, random_state=42))
ridge.fit(X_train, y_train)
pred_val = ridge.predict(X_val)
pred_test = ridge.predict(X_test)
report_mae(y_val, pred_val, "Ridge val MAE")
mae_test_ridge = report_mae(y_test, pred_test, "Ridge test MAE")

for j, name in enumerate(TARGETS):
    print(name, "R2 test", r2_score(y_test[:, j], pred_test[:, j]))

基线 MAE (train mean) {'stars': 0.677764, 'target_food': 0.6731494849173734, 'target_service': 0.6702398803907974, 'target_atmosphere': 0.6989705167329778, 'target_value': 0.676572885016259, 'macro': 0.6793393534114816}
Ridge val MAE {'stars': 0.5679805996548049, 'target_food': 0.5732249232930765, 'target_service': 0.5769481938687068, 'target_atmosphere': 0.5804498605287409, 'target_value': 0.5876985335718338, 'macro': 0.5772604221834325}
Ridge test MAE {'stars': 0.5952248087748495, 'target_food': 0.5894539651973653, 'target_service': 0.5915003242516276, 'target_atmosphere': 0.6160991767026481, 'target_value': 0.6106253768752394, 'macro': 0.600580730360346}
stars R2 test 0.23014012481337665
target_food R2 test 0.22233290615221235
target_service R2 test 0.21433002246708754
target_atmosphere R2 test 0.21045401423534005
target_value R2 test 0.18104999658789045


## Step 8 — 随机森林（多输出）

**验证点**：与 Ridge 对比 macro MAE；若过强可减小 `n_estimators` 或加深正则。

In [58]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=120,
    max_depth=14,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
rf_test = rf.predict(X_test)
mae_test_rf = report_mae(y_test, rf_test, "RandomForest test MAE")

# sklearn：多输出 y 时 feature_importances_ 仍是一维 (n_features,)，已对多个目标做平均
imp = rf.feature_importances_
assert imp.shape == (X_train.shape[1],), (imp.shape, X_train.shape[1])
n_num = len(NUMERIC_FEATURES)
imp_num = imp[:n_num]
fi = (
    pd.DataFrame({"feature": NUMERIC_FEATURES, "importance": imp_num})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
print("数值特征重要性（其余列为 state one-hot + TF-IDF→SVD）:")
display(fi)
print("总特征维", len(imp), "| 数值以外的重要性合计", float(imp[n_num:].sum()))

RandomForest test MAE {'stars': 0.5755716883116533, 'target_food': 0.567090198998164, 'target_service': 0.5700658571525775, 'target_atmosphere': 0.5920679865042316, 'target_value': 0.578603769090756, 'macro': 0.5766799000114766}
数值特征重要性（其余列为 state one-hot + TF-IDF→SVD）:


,feature,importance
0,review_count,0.062866
1,hours_days_open,0.051286
2,longitude,0.037962
3,latitude,0.034699
4,attr_price_range,0.022995
5,is_open,0.012214
6,categories_count,0.011730
7,rev_high_review_star_share,0.005056
8,attr_outdoor_seating,0.005039
9,attr_delivery,0.004505


总特征维 64 | 数值以外的重要性合计 0.7308971304574442


## Step 8c — XGBoost 改进版

1. **区间样本权重**：按半星档位显式加权（1.0/1.5/5.0 最高，2.0/2.5/4.5 中等，3.0–4.0 默认），见 `multid_prep.star_interval_sample_weight`。
2. **先分类再回归**：用 XGB 三分类预测 **低 / 中 / 高** 档（≤2.5，(2.5,4]，>4），再在训练集上 **分档各自拟合** 多输出回归；样本过少档位回退全量回归。
3. **负面特征**：Step 4 已并入尖锐词密度、单星占比、`feat_neg_concentration` 等。

单模型对照：同权重下额外训练一个「不分档」XGB 多输出，便于对比 MAE。

依赖：`pip install xgboost`。

In [59]:
from sklearn.metrics import mean_absolute_error
from sklearn.multioutput import MultiOutputRegressor

from xgboost import XGBRegressor

# 极端评分加权：远离训练集 stars 中位数的样本权重大（抑制「全往 3.5 挤」）
s_tr = train_df["stars"].to_numpy(dtype=float)
star_med = float(np.median(s_tr))
dist = np.abs(s_tr - star_med)
scale = float(np.median(dist) + 0.35)
sample_w = 1.0 + 1.6 * (dist / scale)
sample_w = np.clip(sample_w, 0.6, 4.0)
print(
    "stars 中位数(训练):",
    star_med,
    "| 权重 min/mean/max:",
    float(sample_w.min()),
    float(sample_w.mean()),
    float(sample_w.max()),
)

xgb_est = XGBRegressor(
    objective="reg:absoluteerror",
    n_estimators=450,
    max_depth=6,
    learning_rate=0.04,
    subsample=0.88,
    colsample_bytree=0.88,
    reg_lambda=1.2,
    min_child_weight=2,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
)

xgb_multi = MultiOutputRegressor(xgb_est)
xgb_multi.fit(X_train, y_train, sample_weight=sample_w)
pred_xgb_test = xgb_multi.predict(X_test)
mae_test_xgb = report_mae(y_test, pred_xgb_test, "XGBoost (MAE+加权) test MAE")


st = test_df["stars"].to_numpy(dtype=float)
buckets = [
    ("≤2.5 低分", st <= 2.5),
    ("(2.5, 3.5] 中段偏低", (st > 2.5) & (st <= 3.5)),
    ("(3.5, 4.5] 中段偏高", (st > 3.5) & (st <= 4.5)),
    (">4.5 高分", st > 4.5),
]
j0 = TARGETS.index("stars")
yt, pr_r, pr_rf, pr_x = (
    y_test[:, j0],
    pred_test[:, j0],
    rf_test[:, j0],
    pred_xgb_test[:, j0],
)

print("── 整体 stars 的 MAE：按真实评分分段（越低越好）──")
bucket_rows = []
for name, mask in buckets:
    m = np.asarray(mask, dtype=bool)
    if not m.any():
        continue
    bucket_rows.append(
        {
            "真实 stars 分段": name,
            "样本数": int(m.sum()),
            "Ridge": float(mean_absolute_error(yt[m], pr_r[m])),
            "RandomForest": float(mean_absolute_error(yt[m], pr_rf[m])),
            "XGBoost": float(mean_absolute_error(yt[m], pr_x[m])),
        }
    )
display(pd.DataFrame(bucket_rows))

xgb_fi_mean = np.mean([est.feature_importances_ for est in xgb_multi.estimators_], axis=0)
fi_xgb_num = pd.DataFrame(
    {"importance": xgb_fi_mean[: len(NUMERIC_FEATURES)]}, index=NUMERIC_FEATURES
).sort_values("importance", ascending=False)
print("XGB 各输出特征重要性的平均（仅可解释数值列）:")
display(fi_xgb_num.head(12))

stars 中位数(训练): 3.5 | 权重 min/mean/max: 1.0 2.231725490196079 4.0
XGBoost (MAE+加权) test MAE {'stars': 0.5521374897956848, 'target_food': 0.549691222704983, 'target_service': 0.5576082899428996, 'target_atmosphere': 0.5710287443419018, 'target_value': 0.5589996206357776, 'macro': 0.5578930734842493}
── 整体 stars 的 MAE：按真实评分分段（越低越好）──


,真实 stars 分段,样本数,Ridge,RandomForest,XGBoost
0,≤2.5 低分,87,1.042344,1.004374,0.961569
1,"(2.5, 3.5] 中段偏低",172,0.363195,0.385847,0.449258
2,"(3.5, 4.5] 中段偏高",215,0.529307,0.498059,0.429054
3,>4.5 高分,26,1.179147,1.036813,0.880512


XGB 各输出特征重要性的平均（仅可解释数值列）:


,importance
rev_high_review_star_share,0.025164
hours_days_open,0.021363
feat_neg_concentration,0.020710
rev_mean_sentiment,0.020316
rev_low_review_star_share,0.020276
rev_one_star_share,0.020078
rev_harsh_word_density,0.019082
rev_neg_density_std,0.018757
rev_harsh_review_share,0.017783
rev_max_neg_density,0.017326


## Step 8b — 结果对比（表格 + 图）

**请先运行 Step 8c（XGBoost）**，再运行本格（需要 `mae_test_xgb`、`pred_xgb_test`）。

- **汇总表**：基线 / Ridge / RandomForest / **XGBoost** 在 test 上的 MAE。
- **柱状图**：HTML 横向条形，四模型对比（MAE 越低越好）。
- **相对基线**：三模型相对训练集均值基线的 MAE 降幅（%）。
- **R²**：含 Ridge / RF / XGB。

若本机 `matplotlib` 损坏，本步仍可不依赖 matplotlib 运行。

In [60]:
from sklearn.metrics import mean_absolute_error
from sklearn.multioutput import MultiOutputRegressor

from xgboost import XGBClassifier, XGBRegressor


def stars_to_tier_vec(s: np.ndarray) -> np.ndarray:
    s = np.asarray(s, dtype=float)
    t = np.ones(len(s), dtype=np.int32)
    t[s <= 2.5] = 0
    t[s > 4.0] = 2
    return t


s_tr = train_df["stars"].to_numpy(dtype=float)
sample_w = mp.star_interval_sample_weight(s_tr)
print(
    "区间样本权重 min/mean/max:",
    float(sample_w.min()),
    float(sample_w.mean()),
    float(sample_w.max()),
)

y_tier_tr = stars_to_tier_vec(s_tr)
print("训练集 低/中/高 档样本数:", [int((y_tier_tr == k).sum()) for k in (0, 1, 2)])

tier_clf = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=280,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    eval_metric="mlogloss",
)
tier_clf.fit(X_train, y_tier_tr, sample_weight=sample_w)

xgb_reg_params = dict(
    objective="reg:absoluteerror",
    n_estimators=380,
    max_depth=5,
    learning_rate=0.045,
    subsample=0.88,
    colsample_bytree=0.88,
    reg_lambda=1.2,
    min_child_weight=2,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
)

MIN_TIER_N = 28
tier_regs = {}
for t in (0, 1, 2):
    m = y_tier_tr == t
    n = int(m.sum())
    if n < MIN_TIER_N:
        print(f"档 {t} 训练样本 {n} < {MIN_TIER_N}，回归使用全量 fallback")
        tier_regs[t] = None
    else:
        reg = MultiOutputRegressor(XGBRegressor(**xgb_reg_params))
        reg.fit(X_train[m], y_train[m], sample_weight=sample_w[m])
        tier_regs[t] = reg

xgb_fallback = MultiOutputRegressor(XGBRegressor(**xgb_reg_params))
xgb_fallback.fit(X_train, y_train, sample_weight=sample_w)

tier_te = tier_clf.predict(X_test)
pred_xgb_test = np.zeros_like(y_test)
for t in (0, 1, 2):
    mask = tier_te == t
    if not np.any(mask):
        continue
    r = tier_regs[t]
    if r is not None:
        pred_xgb_test[mask] = r.predict(X_test[mask])
    else:
        pred_xgb_test[mask] = xgb_fallback.predict(X_test[mask])

mae_test_xgb = report_mae(y_test, pred_xgb_test, "XGBoost（分档回归+区间权重） test MAE")

y_tier_te = stars_to_tier_vec(test_df["stars"].to_numpy(dtype=float))
print("分档分类 accuracy（test）:", float(np.mean(tier_te == y_tier_te)))

flat_params = {**xgb_reg_params, "n_estimators": 420}
xgb_flat = MultiOutputRegressor(XGBRegressor(**flat_params))
xgb_flat.fit(X_train, y_train, sample_weight=sample_w)
pred_xgb_flat_test = xgb_flat.predict(X_test)
_ = report_mae(y_test, pred_xgb_flat_test, "XGBoost（单模型+区间权重，对照） test MAE")

st = test_df["stars"].to_numpy(dtype=float)
buckets = [
    ("≤2.5 低分", st <= 2.5),
    ("(2.5, 3.5] 中段偏低", (st > 2.5) & (st <= 3.5)),
    ("(3.5, 4.5] 中段偏高", (st > 3.5) & (st <= 4.5)),
    (">4.5 高分", st > 4.5),
]
j0 = TARGETS.index("stars")
yt, pr_r, pr_rf, pr_x = (
    y_test[:, j0],
    pred_test[:, j0],
    rf_test[:, j0],
    pred_xgb_test[:, j0],
)

print("── 整体 stars MAE 按真实分段 ──")
bucket_rows = []
for name, mask in buckets:
    m = np.asarray(mask, dtype=bool)
    if not m.any():
        continue
    bucket_rows.append(
        {
            "真实 stars 分段": name,
            "样本数": int(m.sum()),
            "Ridge": float(mean_absolute_error(yt[m], pr_r[m])),
            "RandomForest": float(mean_absolute_error(yt[m], pr_rf[m])),
            "XGB分档": float(mean_absolute_error(yt[m], pr_x[m])),
            "XGB单模型": float(mean_absolute_error(yt[m], pred_xgb_flat_test[:, j0][m])),
        }
    )
display(pd.DataFrame(bucket_rows))

xgb_fi_mean = np.mean([est.feature_importances_ for est in xgb_flat.estimators_], axis=0)
fi_xgb_num = pd.DataFrame(
    {"importance": xgb_fi_mean[: len(NUMERIC_FEATURES)]}, index=NUMERIC_FEATURES
).sort_values("importance", ascending=False)
print("特征重要性（单模型 XGB 各输出平均，数值列 Top）：")
display(fi_xgb_num.head(14))


区间样本权重 min/mean/max: 1.0 1.4376 3.2
训练集 低/中/高 档样本数: [264, 918, 318]
XGBoost（分档回归+区间权重） test MAE {'stars': 0.57449742436409, 'target_food': 0.5576610584354857, 'target_service': 0.5705378001279666, 'target_atmosphere': 0.59789178124065, 'target_value': 0.5895587836931007, 'macro': 0.5780293695722586}
分档分类 accuracy（test）: 0.674
XGBoost（单模型+区间权重，对照） test MAE {'stars': 0.5361106123924255, 'target_food': 0.5411234437681914, 'target_service': 0.5555883464094838, 'target_atmosphere': 0.561950877498912, 'target_value': 0.5562276571877082, 'macro': 0.5502001874513442}
── 整体 stars MAE 按真实分段 ──


,真实 stars 分段,样本数,Ridge,RandomForest,XGB分档,XGB单模型
0,≤2.5 低分,87,1.042344,1.004374,0.897338,0.951196
1,"(2.5, 3.5] 中段偏低",172,0.363195,0.385847,0.473742,0.432757
2,"(3.5, 4.5] 中段偏高",215,0.529307,0.498059,0.489378,0.410739
3,>4.5 高分,26,1.179147,1.036813,0.864632,0.867622


特征重要性（单模型 XGB 各输出平均，数值列 Top）：


,importance
hours_days_open,0.028390
rev_high_review_star_share,0.024464
feat_neg_concentration,0.019520
rev_mean_sentiment,0.019331
rev_low_review_star_share,0.018426
rev_one_star_share,0.017609
rev_harsh_word_density,0.017448
rev_neg_density_std,0.016341
is_open,0.016193
rev_harsh_review_share,0.016148


## Step 8d — 分类指标（Precision / Recall / F1）

回归模型输出的是连续分（1–5）。这里把 **真实 / 预测 stars 四舍五入到最近半星**，当作 **9 类分类**（1.0, 1.5, …, 5.0），再计算 `precision`、`recall`、`F1`（macro / weighted）。

另给 **二分类**：「高分」= 真实半星 ≥ 4.0，看模型是否把店判成高分（**precision/recall 针对正类「高分」**）。

**说明**：类别极不平衡时 macro 与 weighted 差异会很大；可与混淆矩阵对照。

In [61]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

j_stars = TARGETS.index("stars")
y_star = y_test[:, j_stars]

STAR_GRID = np.arange(1.0, 5.01, 0.5)
STAR_LABELS = [f"{x:.1f}" for x in STAR_GRID]


def stars_continuous_to_label(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y, dtype=float)
    h = np.round((y - 1.0) * 2) / 2 + 1.0
    h = np.clip(h, 1.0, 5.0)
    return np.array([f"{v:.1f}" for v in h])


def clf_metrics_table(y_true: np.ndarray, y_pred: np.ndarray, model_name: str) -> None:
    yt = stars_continuous_to_label(y_true)
    yp = stars_continuous_to_label(y_pred)
    print(f"\n{'='*60}\n■ {model_name} — 半星多分类（test）\n{'='*60}")
    print("Accuracy:", f"{accuracy_score(yt, yp):.4f}")
    p_m, r_m, f_m, _ = precision_recall_fscore_support(
        yt, yp, labels=STAR_LABELS, average="macro", zero_division=0
    )
    p_w, r_w, f_w, _ = precision_recall_fscore_support(
        yt, yp, labels=STAR_LABELS, average="weighted", zero_division=0
    )
    print(
        f"Macro    — precision={p_m:.4f}  recall={r_m:.4f}  F1={f_m:.4f}\n"
        f"Weighted — precision={p_w:.4f}  recall={r_w:.4f}  F1={f_w:.4f}"
    )
    print("\n逐类报告（support=该真实类在 test 中条数）:")
    print(
        classification_report(
            yt, yp, labels=STAR_LABELS, digits=3, zero_division=0
        )
    )
    cm = confusion_matrix(yt, yp, labels=STAR_LABELS)
    cm_df = pd.DataFrame(cm, index=[f"真 {s}" for s in STAR_LABELS], columns=[f"预 {s}" for s in STAR_LABELS])
    print("混淆矩阵（行=真实，列=预测）:")
    display(cm_df)


def high_star_binary_metrics(y_true: np.ndarray, y_pred: np.ndarray, model_name: str) -> None:
    """正类：真实半星 >= 4.0 为「高分店」。"""
    yt = stars_continuous_to_label(y_true).astype(float) >= 4.0
    yp = stars_continuous_to_label(y_pred).astype(float) >= 4.0
    p, r, f, _ = precision_recall_fscore_support(
        yt, yp, average="binary", pos_label=True, zero_division=0
    )
    print(
        f"{model_name} — 二分类「真实≥4星为高分」: "
        f"precision={p:.4f}  recall={r:.4f}  F1={f:.4f}  "
        f"(正类占比 真={yt.mean():.3f} 预={yp.mean():.3f})"
    )


print("── 整体 stars：分类视角（需已运行 Ridge / RF / XGB 预测格）──")
for name, pred in [
    ("Ridge", pred_test[:, j_stars]),
    ("RandomForest", rf_test[:, j_stars]),
    ("XGBoost", pred_xgb_test[:, j_stars]),
]:
    clf_metrics_table(y_star, pred, name)

print("\n── 二分类：是否「高分店」(真实≥4星) ──")
for name, pred in [
    ("Ridge", pred_test[:, j_stars]),
    ("RandomForest", rf_test[:, j_stars]),
    ("XGBoost", pred_xgb_test[:, j_stars]),
]:
    high_star_binary_metrics(y_star, pred, name)

── 整体 stars：分类视角（需已运行 Ridge / RF / XGB 预测格）──

■ Ridge — 半星多分类（test）
Accuracy: 0.2720
Macro    — precision=0.1958  recall=0.1590  F1=0.1503
Weighted — precision=0.2764  recall=0.2720  F1=0.2399

逐类报告（support=该真实类在 test 中条数）:
              precision    recall  f1-score   support

         1.0      0.000     0.000     0.000         1
         1.5      0.000     0.000     0.000        18
         2.0      0.333     0.083     0.133        24
         2.5      0.143     0.091     0.111        44
         3.0      0.224     0.242     0.233        62
         3.5      0.264     0.564     0.359       110
         4.0      0.298     0.321     0.309       131
         4.5      0.500     0.131     0.208        84
         5.0      0.000     0.000     0.000        26

    accuracy                          0.272       500
   macro avg      0.196     0.159     0.150       500
weighted avg      0.276     0.272     0.240       500

混淆矩阵（行=真实，列=预测）:


,预 1.0,预 1.5,预 2.0,预 2.5,预 3.0,预 3.5,预 4.0,预 4.5,预 5.0
真 1.0,0,0,1,0,0,0,0,0,0
真 1.5,0,0,2,4,4,7,1,0,0
真 2.0,0,0,2,10,4,8,0,0,0
真 2.5,0,0,0,4,10,16,13,1,0
真 3.0,0,0,1,3,15,32,10,1,0
真 3.5,0,0,0,5,12,62,28,3,0
真 4.0,0,0,0,1,18,67,42,3,0
真 4.5,0,0,0,1,4,30,37,11,1
真 5.0,0,0,0,0,0,13,10,3,0



■ RandomForest — 半星多分类（test）
Accuracy: 0.2880
Macro    — precision=0.2055  recall=0.1718  F1=0.1616
Weighted — precision=0.2760  recall=0.2880  F1=0.2438

逐类报告（support=该真实类在 test 中条数）:
              precision    recall  f1-score   support

         1.0      0.000     0.000     0.000         1
         1.5      0.000     0.000     0.000        18
         2.0      0.455     0.208     0.286        24
         2.5      0.194     0.136     0.160        44
         3.0      0.204     0.161     0.180        62
         3.5      0.283     0.527     0.368       110
         4.0      0.314     0.466     0.375       131
         4.5      0.400     0.048     0.085        84
         5.0      0.000     0.000     0.000        26

    accuracy                          0.288       500
   macro avg      0.206     0.172     0.162       500
weighted avg      0.276     0.288     0.244       500

混淆矩阵（行=真实，列=预测）:


,预 1.0,预 1.5,预 2.0,预 2.5,预 3.0,预 3.5,预 4.0,预 4.5,预 5.0
真 1.0,0,0,0,1,0,0,0,0,0
真 1.5,0,0,3,6,1,6,2,0,0
真 2.0,0,0,5,6,5,8,0,0,0
真 2.5,0,0,2,6,8,13,15,0,0
真 3.0,0,0,0,4,10,32,16,0,0
真 3.5,0,0,1,4,10,58,36,1,0
真 4.0,0,0,0,3,11,56,61,0,0
真 4.5,0,0,0,1,4,24,51,4,0
真 5.0,0,0,0,0,0,8,13,5,0



■ XGBoost — 半星多分类（test）
Accuracy: 0.3100
Macro    — precision=0.2721  recall=0.2213  F1=0.2076
Weighted — precision=0.3332  recall=0.3100  F1=0.2737

逐类报告（support=该真实类在 test 中条数）:
              precision    recall  f1-score   support

         1.0      0.000     0.000     0.000         1
         1.5      0.333     0.167     0.222        18
         2.0      0.163     0.292     0.209        24
         2.5      0.333     0.136     0.194        44
         3.0      0.286     0.032     0.058        62
         3.5      0.282     0.673     0.398       110
         4.0      0.413     0.198     0.268       131
         4.5      0.389     0.417     0.402        84
         5.0      0.250     0.077     0.118        26

    accuracy                          0.310       500
   macro avg      0.272     0.221     0.208       500
weighted avg      0.333     0.310     0.274       500

混淆矩阵（行=真实，列=预测）:


,预 1.0,预 1.5,预 2.0,预 2.5,预 3.0,预 3.5,预 4.0,预 4.5,预 5.0
真 1.0,0,0,1,0,0,0,0,0,0
真 1.5,0,3,6,3,0,4,0,2,0
真 2.0,0,4,7,3,1,9,0,0,0
真 2.5,0,1,9,6,1,19,4,4,0
真 3.0,0,0,9,1,2,41,4,4,1
真 3.5,0,1,7,1,3,74,12,11,1
真 4.0,0,0,3,3,0,78,26,21,0
真 4.5,0,0,1,0,0,30,14,35,4
真 5.0,0,0,0,1,0,7,3,13,2



── 二分类：是否「高分店」(真实≥4星) ──
Ridge — 二分类「真实≥4星为高分」: precision=0.6524  recall=0.4440  F1=0.5284  (正类占比 真=0.482 预=0.328)
RandomForest — 二分类「真实≥4星为高分」: precision=0.6569  recall=0.5560  F1=0.6022  (正类占比 真=0.482 预=0.408)
XGBoost — 二分类「真实≥4星为高分」: precision=0.7329  recall=0.4896  F1=0.5871  (正类占比 真=0.482 预=0.322)


## Step 9 — 样例解释（可选）

查看某商户的预测 vs 真实弱标签，对照评论是否说得通。

In [62]:
sample = test_df.iloc[:5][["business_id", "name", "stars"] + [c for c in TARGETS if c != "stars"]]
pred_s = ridge.predict(X_test[:5])
out = pd.DataFrame(pred_s, columns=[f"pred_{t}" for t in TARGETS])
pd.concat([sample.reset_index(drop=True), out], axis=1)

,business_id,name,stars,target_food,target_service,target_atmosphere,target_value,pred_stars,pred_target_food,pred_target_service,pred_target_atmosphere,pred_target_value
0,WzWBfCHurRUGpQuXkMnfow,Copa Cabana,2.0,2.925000,4.12500,2.00,2.775000,3.055728,3.481906,3.679376,3.472400,3.163715
1,A3FD_WGaUBASqbR8ID6ZUA,Subway,2.0,2.000000,2.00000,2.00,2.000000,3.313937,3.340167,3.283370,3.310638,3.270824
2,1AJHVnRGxTWNMaFbIUbWcg,Starbucks,2.5,2.500000,2.50000,2.50,2.500000,3.504484,3.555373,3.517201,3.537366,3.494671
3,TnE88JiuC6mzeejgl7yjnA,Fionn MacCool's Irish Pub,2.5,2.500000,2.50000,2.50,2.500000,3.795496,3.825879,3.762770,3.818798,3.759646
4,wBxTjmTpT8E0s6DIcaaDCA,Dumplings the Restaurant,3.0,3.485238,3.16381,1.84,3.674286,3.177906,3.654975,3.431588,3.656374,3.656824
